# Myproject — Train 100 món ăn trên Google Colab

**Mục tiêu:** Mở rộng 50 → 100 món (giữ nguyên thứ tự 50 nhãn cũ, thêm 50 món Food-101).

**Trước khi chạy:** chọn **Runtime → Change runtime type → T4 GPU**. Dùng tài khoản Google Drive có đủ dung lượng cho checkpoint.

Notebook lấy **30VNFoods** từ Kaggle (quandang/vietnamese-foods), tải **Food-101** bằng torchvision, huấn luyện EfficientNet-B0 và xuất ONNX. Dataset có thể lớn và Colab có thể ngắt phiên: checkpoint sẽ được lưu vào Google Drive.

**Quan trọng:** Notebook này KHÔNG thay model đang chạy trên Render/Vercel; chỉ tạo các file mới để xem kết quả và kiểm định trước khi triển khai.


In [ ]:
# 1 — Cài thư viện
%pip -q install "kagglehub>=0.3" scikit-learn onnx onnxruntime onnxscript


In [ ]:
# 2 — Kết nối Drive và kiểm tra GPU
from google.colab import drive
drive.mount("/content/drive")

import os, sys, subprocess, shutil, json, zipfile
from pathlib import Path

gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
print(gpu.stdout.strip() or "Không tìm thấy GPU — hãy chọn T4 GPU trong Runtime")
import torch
print("CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Hãy bật Runtime → Change runtime type → T4 GPU trước khi train."

OUTPUT = Path("/content/drive/MyDrive/Myproject_food100")
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Kết quả sẽ lưu vào:", OUTPUT)


## Tải dữ liệu 30 món Việt

Mặc định dùng **KaggleHub** tải bộ `quandang/vietnamese-foods` trực tiếp vào Colab; không cần `/kaggle/input`.

Nếu Kaggle yêu cầu đăng nhập/quota hoặc tải thất bại, chuyển `VN_SOURCE` thành `"drive_zip"` (đặt file ZIP trong MyDrive) hoặc `"drive_folder"` (thư mục đã giải nén trên Drive).

Cấu trúc yêu cầu: `Images/Train/<tên món>/*.jpg`, `Images/Validate/<tên món>/*.jpg`, `Images/Test/<tên món>/*.jpg`; cả ba split chứa cùng 30 nhãn như lần train trước.


In [ ]:
# 3 — Chọn nguồn ảnh Việt Nam (mặc định tải trực tiếp Kaggle)
import kagglehub

VN_SOURCE = "kagglehub"  # "kagglehub", "drive_zip", hoặc "drive_folder"
DRIVE_ZIP = Path("/content/drive/MyDrive/vietnamese-foods.zip")
DRIVE_FOLDER = Path("/content/drive/MyDrive/vietnamese-foods")

def locate_vietnamese_root(root):
    root = Path(root)
    candidates = [root / "Images", root]
    if root.exists():
        candidates += [p.parent for p in root.rglob("Train") if p.is_dir()]
    for candidate in candidates:
        if all((candidate / split).is_dir() for split in ("Train", "Validate", "Test")):
            return candidate
    raise FileNotFoundError(
        f"Không tìm thấy bộ 3 thư mục Train/Validate/Test trong {root}. "
        "Kiểm tra lại VN_SOURCE hoặc đường dẫn dữ liệu."
    )

if VN_SOURCE == "kagglehub":
    try:
        vn_download = Path(kagglehub.dataset_download("quandang/vietnamese-foods"))
    except Exception as exc:
        raise RuntimeError(
            "Không tải được Kaggle. Đổi VN_SOURCE sang 'drive_zip' hoặc "
            "'drive_folder' sau khi đặt dataset trong Google Drive."
        ) from exc
elif VN_SOURCE == "drive_zip":
    if not DRIVE_ZIP.is_file():
        raise FileNotFoundError(f"Đưa ZIP vào Google Drive trước: {DRIVE_ZIP}")
    vn_download = Path("/content/vietnamese-foods")
    vn_download.mkdir(parents=True, exist_ok=True)
    # Chỉ dùng ZIP tải từ nguồn bạn tin tưởng.
    with zipfile.ZipFile(DRIVE_ZIP) as archive:
        destination = vn_download.resolve()
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if not target.is_relative_to(destination):
                raise ValueError("ZIP chứa đường dẫn không an toàn")
        archive.extractall(destination)
elif VN_SOURCE == "drive_folder":
    vn_download = DRIVE_FOLDER
else:
    raise ValueError("VN_SOURCE không hợp lệ")

VN_ROOT = locate_vietnamese_root(vn_download)
print("Vietnamese data found:", VN_ROOT)
for split in ("Train", "Validate", "Test"):
    folder = VN_ROOT / split
    print(split, "classes:", len([p for p in folder.iterdir() if p.is_dir()]))
assert len([p for p in (VN_ROOT / "Train").iterdir() if p.is_dir()]) == 30, "Cần đủ 30 món Việt Nam"


## Lấy code Myproject

Colab sẽ tải riêng nhánh `feature/food100-training`, không chỉnh `main`, `master` hay model đang được deploy.


In [ ]:
# 4 — Tải script train từ đúng branch GitHub
REPO = Path("/content/Myproject_food100_repo")
if not (REPO / "ml" / "train_food_100.py").is_file():
    subprocess.run([
        "git", "clone", "--depth", "1", "--branch", "feature/food100-training",
        "https://github.com/Tl4826/Myproject.git", str(REPO)
    ], check=True)
TRAIN_SCRIPT = REPO / "ml" / "train_food_100.py"
assert TRAIN_SCRIPT.exists(), f"Missing: {TRAIN_SCRIPT}"
print("Code:", TRAIN_SCRIPT)


## Chuẩn bị model 50 món (tùy chọn)

Nếu bạn còn **checkpoint PyTorch thật** từ lần train 50 món (`.pt` hoặc `.pth`), upload vào Drive rồi điền `OLD_CHECKPOINT`. Nếu checkpoint chưa chứa tên lớp, điền thêm `OLD_CLASSES` bằng file JSON chính xác từ lần train 50 món.

Nếu không còn checkpoint, cứ để `None`: script sẽ train 100 món từ EfficientNet-B0 pretrained trên ImageNet; **không thể dùng ONNX 12 món đang deploy để tiếp tục train 50→100**.


In [ ]:
# 5 — Điền checkpoint cũ (nếu có), hoặc để None
OLD_CHECKPOINT = None  # Ví dụ: "/content/drive/MyDrive/food50/best.pt"
OLD_CLASSES = None     # Ví dụ: "/content/drive/MyDrive/food50/classes.json"

FOOD_ROOT = Path("/content/food101_data")  # torchvision sẽ tải Food-101 vào đây
EPOCHS_HEAD = 2
EPOCHS_FULL = 8
BATCH_SIZE = 32

# Nếu không có đủ RAM GPU, giảm BATCH_SIZE thành 16.


## Train model 100 món

Lần đầu chạy, Food-101 sẽ được tải về Colab (khoảng **5 GB** dữ liệu nén) và giải nén. Toàn bộ **100 lớp** phải có ảnh Train/Validate/Test. Model tốt nhất theo validation macro-F1 lưu vào Drive. Tốc độ tùy GPU/dung lượng.

Không đóng hoặc ngắt Colab giữa lúc train.


In [ ]:
# 6 — Bắt đầu huấn luyện (GPU)
cmd = [
    sys.executable, str(TRAIN_SCRIPT),
    "--vn-root", str(VN_ROOT),
    "--food-root", str(FOOD_ROOT),
    "--download-food101",
    "--out", str(OUTPUT),
    "--epochs-head", str(EPOCHS_HEAD),
    "--epochs-full", str(EPOCHS_FULL),
    "--batch-size", str(BATCH_SIZE),
    "--max-food-train", "350",
]
if OLD_CHECKPOINT:
    cmd.extend(["--old-checkpoint", str(OLD_CHECKPOINT)])
if OLD_CLASSES:
    cmd.extend(["--old-classes", str(OLD_CLASSES)])
print("Training 100 classes. Checkpoint saved to", OUTPUT)
subprocess.run(cmd, check=True)


## Đọc kết quả

Chỉ khi huấn luyện hoàn thành thì mới có metrics trên bộ test và file `food_classifier.onnx`.

Không ghi đè `ml/models/food_classifier.onnx` đang triển khai cho đến khi kiểm tra API, nhãn và nutrition lookup hỗ trợ đủ 100 món.


In [ ]:
# 7 — Đánh giá và nén các file cần triển khai
metrics_file = OUTPUT / "test_metrics.json"
onnx_file = OUTPUT / "food_classifier.onnx"
labels_file = OUTPUT / "classes.json"

assert metrics_file.exists() and onnx_file.exists() and labels_file.exists(), (
    "Chưa train xong; kiểm tra lỗi ở cell 6."
)
metrics = json.loads(metrics_file.read_text(encoding="utf-8"))
classes = json.loads(labels_file.read_text(encoding="utf-8"))["classes"]
assert len(classes) == len(set(classes)) == 100
print("TEST RESULTS:", json.dumps(metrics, indent=2))
print("ONNX:", onnx_file, onnx_file.stat().st_size, "bytes")
print("Classes:", len(classes))

PACKAGE = OUTPUT / "myproject_food100_deploy.zip"
with zipfile.ZipFile(PACKAGE, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(onnx_file, arcname="food_classifier.onnx")
    z.write(labels_file, arcname="classes.json")
    z.write(metrics_file, arcname="test_metrics.json")
print("Saved model package:", PACKAGE)
